In [ ]:
import numpy as np
import math
from keras.models import Sequential
from keras.layers import Dense, LSTM, Dropout
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
from tensorflow.keras.optimizers import Adam

def lstm_estimator_past(data, N_remove, N_estimate, link, normalize_data=True, epochs=50, save_figures=False, save_path=None, show_figs=False):
    print(f'Estimating for link {link}')

    # Data preparation as before
    dates = data['Date']
    data['DayOfWeek'] = data['Date'].dt.dayofweek
    data['Month'] = data['Date'].dt.month
    feature_columns = ['Error', 'DayOfWeek', 'Month']
    data_features = data[feature_columns]

    if normalize_data:
        scaler = MinMaxScaler(feature_range=(0, 1))
        data_values = scaler.fit_transform(data_features)
    else:
        data_values = data_features.values

    # Split the data
    N = len(data_values)
    if N_remove == 0:
        past_data_values = data_values
        past_dates = dates
        last_date = past_dates.iloc[-1]
        future_dates = pd.date_range(start=last_date, periods=N_estimate + 1, freq='D')[1:]
    else:
        past_data_values = data_values[:-N_remove]
        future_data_values = data_values[-N_remove:]
        past_dates = dates[:-N_remove]
        future_dates = pd.date_range(start=dates.iloc[-N_remove], periods=N_estimate + 1, freq='D')[1:]

    # Create sequences for LSTM
    sequence_length = 60
    def create_sequences(data, sequence_length):
        sequences = []
        labels = []
        for i in range(sequence_length, len(data)):
            sequences.append(data[i-sequence_length:i])
            labels.append(data[i, 0])  # Predicting the 'Error' value (first column)
        return np.array(sequences), np.array(labels)

    X, y = create_sequences(past_data_values, sequence_length)

    # Split into training and testing data
    split = int(0.8 * len(X))
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    # Define the LSTM model using Input layer
    model = Sequential()
    
    # Add Input layer to define the input shape
    model.add(Input(shape=(X_train.shape[1], X_train.shape[2])))
    
    # Add LSTM layers
    model.add(LSTM(150, return_sequences=True))
    model.add(LSTM(150))
    model.add(Dense(1))

    # Compile the model
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')

    # Train the model
    history = model.fit(X_train, y_train, epochs=epochs, batch_size=32, validation_data=(X_test, y_test), verbose=0)

    # Predict past data
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Inverse transform if data was normalized
    if normalize_data:
        y_train_pred = scaler.inverse_transform(np.concatenate([y_train_pred, X_train[:, -1, 1:]], axis=1))[:, 0]
        y_test_pred = scaler.inverse_transform(np.concatenate([y_test_pred, X_test[:, -1, 1:]], axis=1))[:, 0]
        y_train = scaler.inverse_transform(np.concatenate([y_train.reshape(-1, 1), X_train[:, -1, 1:]], axis=1))[:, 0]
        y_test = scaler.inverse_transform(np.concatenate([y_test.reshape(-1, 1), X_test[:, -1, 1:]], axis=1))[:, 0]
    else:
        y_train_pred = y_train_pred
        y_test_pred = y_test_pred

    last_sequence = past_data_values[-sequence_length:].reshape((1, sequence_length, past_data_values.shape[1]))
    future_predictions = []

    for _ in range(N_estimate):
        future_pred = model.predict(last_sequence)[0]  # Make prediction

        if future_pred.size == 0:  # Check if prediction is valid
            break  # Avoid continuing if predictions are not generated

        future_predictions.append(future_pred)  # Append prediction to future_predictions

        # Update last_sequence for the next prediction step
        last_sequence = np.append(last_sequence[:, 1:, :], [[np.concatenate([future_pred, [last_sequence[0, -1, 1], last_sequence[0, -1, 2]]])]], axis=1)
        
    # Check if future_predictions contains data
    if len(future_predictions) == 0:
        raise ValueError("No future predictions were generated.")

    future_predictions = np.array(future_predictions).reshape(-1, 1)

    # Inverse transform future predictions if data was normalized
    if normalize_data:
        future_predictions = scaler.inverse_transform(np.concatenate([future_predictions, np.zeros((future_predictions.shape[0], 2))], axis=1))[:, 0]

    # Combine train and test predictions
    complete_dates = pd.concat([pd.Series(dates[:-N_remove]), pd.Series(future_dates)])
    complete_estimated_data = np.concatenate([model.predict(X).flatten(), future_predictions.flatten()])

    # Error metrics
    error_of_train_estimation = np.abs(y_train - y_train_pred)
    mean_error_train_estimation = np.mean(error_of_train_estimation)
    std_error_train_estimation = np.std(error_of_train_estimation)
    rmse_error_train_estimation = np.sqrt(mean_squared_error(y_train, y_train_pred))

    error_of_test_estimation = np.abs(y_test - y_test_pred)
    mean_error_test_estimation = np.mean(error_of_test_estimation)
    std_error_test_estimation = np.std(error_of_test_estimation)
    rmse_error_test_estimation = np.sqrt(mean_squared_error(y_test, y_test_pred))

    result = {
        link: {
            "Normalized_data": normalize_data,

            "Complete_dates": complete_dates,
            "Complete_estimated_data": complete_estimated_data.tolist(),
                       
            "Past_dates": dates,
            "Last_date": dates.iloc[-1],
            "Past_data": data['Error'].values.tolist(),

            "Future_dates": future_dates,
            "Future_estimated_data": future_predictions.tolist(),

            "Train_dates": dates[:split + sequence_length],
            "Last_train_date": dates[split + sequence_length - 1],
            "Train_data": y_train.tolist(),
            "Train_estimated_data": y_train_pred.tolist(),

            "Train_estimation_error": error_of_train_estimation.tolist(),
            "Train_mean_estimation_error": mean_error_train_estimation,
            "Train_std_estimation_error": std_error_train_estimation,
            "Train_rmse_estimation_error": rmse_error_train_estimation,

            "Test_dates": dates[split + sequence_length:],
            "Test_data": y_test.tolist(),
            "Test_estimated_data": y_test_pred.tolist(),

            "Test_estimation_error": error_of_test_estimation.tolist(),
            "Test_mean_estimation_error": mean_error_test_estimation,
            "Test_std_estimation_error": std_error_test_estimation,
            "Test_rmse_estimation_error": rmse_error_test_estimation,
        }
    }

    if show_figs or save_figures:
        plt.figure(figsize=(18, 30))

        # Train data plot
        plt.subplot(4, 1, 1)
        plt.plot(result[link]['Train_dates'][:len(result[link]['Train_data'])], result[link]['Train_data'], label='Actual Data', color='royalblue')
        plt.plot(result[link]['Train_dates'][:len(result[link]['Train_estimated_data'])], result[link]['Train_estimated_data'], label='Estimated Data', color='darkorange', linestyle='--')
        plt.title(f'LSTM\nActual vs Estimated Data (train)')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.grid()

        # Test data plot
        plt.subplot(4, 1, 2)
        plt.plot(result[link]['Test_dates'][:len(result[link]['Test_data'])], result[link]['Test_data'], label='Actual Data', color='royalblue', marker='o')
        plt.plot(result[link]['Test_dates'][:len(result[link]['Test_estimated_data'])], result[link]['Test_estimated_data'], label='Estimated Data', color='darkorange', linestyle='--', marker='o')
        plt.title(f'Actual vs Estimated Data (test)  RMSE: {result[link]["Test_rmse_estimation_error"]}')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.grid()

        # Upper and lower bounds for test data
        plt.subplot(4, 1, 3)
        plt.plot(result[link]['Test_dates'][:len(result[link]['Test_data'])], np.array(result[link]['Test_data']) + result[link]['Test_mean_estimation_error'], label='Upper Data Bound', color='teal', marker='o')
        plt.plot(result[link]['Test_dates'][:len(result[link]['Test_data'])], np.array(result[link]['Test_data']) - result[link]['Test_mean_estimation_error'], label='Lower Data Bound', color='orchid', marker='o')
        plt.plot(result[link]['Test_dates'][:len(result[link]['Test_estimated_data'])], result[link]['Test_estimated_data'], label='Estimated Data', color='chocolate', linestyle='--', marker='o')
        plt.title(f'Bounds of Actual vs Estimated Data (test) std: {result[link]["Test_std_estimation_error"]}')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.grid()

        plt.tight_layout()
        if save_figures:
            plt.savefig(save_path)

        if show_figs:
            plt.show()

        plt.close()


    return result


In [ ]:
file_path = 'up_to_15-07-2024/brisbane_0-14.xlsx'  # Update this path to the correct location of your Excel file
df = pd.read_excel(file_path)
lstm_estimator_past(df, 10, 10, '0-14', normalize_data=True, epochs=5, save_figures=False, save_path=None, show_figs=True)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, root_mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
from tensorflow.keras.optimizers import Adam

def lstm_estimator(data, days_to_remove, days_to_estimate, link, normalize_data=True, epochs=50, save_figures=False, save_path=None, show_figs=False):
    print(f'Estimating for link {link}')
        
    # Data preparation
    data['DayOfWeek'] = data['Date'].dt.dayofweek
    data['Month'] = data['Date'].dt.month
    feature_columns = ['Error', 'DayOfWeek', 'Month']
    data_features = data[feature_columns]

    if normalize_data:
        scaler = MinMaxScaler(feature_range=(0, 1))
        data_values = scaler.fit_transform(data_features)
    else:
        data_values = data_features.values

    N = len(data['Error'])
    
    if days_to_remove == 0:
        past_data_values = data_values
        past_dates = data['Date']
        last_date = past_dates.iloc[-1]
        future_dates = pd.date_range(start=last_date, periods=days_to_estimate + 1, freq='D')[1:]
    else:
        past_data_values = data_values[:-days_to_remove]
        future_data_values = data_values[-days_to_remove:]
        past_dates = data['Date'][:-days_to_remove]
        future_dates = data['Date'][-days_to_remove:]

    # Create sequences for LSTM
    sequence_length = 30

    def create_sequences(data, sequence_length):
        sequences = []
        labels = []
        for i in range(sequence_length, len(data)):
            sequences.append(data[i-sequence_length:i])
            labels.append(data[i, 0])  # Predicting the 'Error' value (first column)
        return np.array(sequences), np.array(labels)

    # Use the past data for training
    X_train, y_train = create_sequences(past_data_values, sequence_length)

    # Use the future data for testing (check if there's enough data to create sequences)
    if days_to_remove >= sequence_length:
        X_test, y_test = create_sequences(future_data_values, sequence_length)
    else:
        # If not enough data, skip the test prediction
        X_test, y_test = np.empty((0, sequence_length, past_data_values.shape[1])), np.empty((0,))
        print(f"Warning: Not enough data in 'days_to_remove' to create test sequences. Skipping test predictions.")

    # Define the LSTM model
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1], X_train.shape[2])))
    model.add(LSTM(150, return_sequences=True))
    model.add(LSTM(150))
    model.add(Dense(1))

    # Compile the model
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')

    # Train the model on past data
    model.fit(X_train, y_train, epochs=epochs, batch_size=32, verbose=1)

    # Predict past data (train set)
    y_train_pred = model.predict(X_train)

    # Inverse transform if data was normalized
    if normalize_data:
        y_train_pred = scaler.inverse_transform(np.concatenate([y_train_pred, X_train[:, -1, 1:]], axis=1))[:, 0]
        y_train = scaler.inverse_transform(np.concatenate([y_train.reshape(-1, 1), X_train[:, -1, 1:]], axis=1))[:, 0]

    # Predict future data (test set) based on the days_to_remove, if valid sequences exist
    if X_test.shape[0] > 0:
        y_test_pred = model.predict(X_test)
        if normalize_data:
            y_test_pred = scaler.inverse_transform(np.concatenate([y_test_pred, X_test[:, -1, 1:]], axis=1))[:, 0]
            y_test = scaler.inverse_transform(np.concatenate([y_test.reshape(-1, 1), X_test[:, -1, 1:]], axis=1))[:, 0]
    else:
        y_test_pred = []

    # Prepare last sequence for future predictions
    last_sequence = past_data_values[-sequence_length:].reshape((1, sequence_length, past_data_values.shape[1]))
    future_predictions = []

    for _ in range(days_to_estimate):
        future_pred = model.predict(last_sequence)[0]
        future_predictions.append(future_pred)
        last_sequence = np.append(last_sequence[:, 1:, :], [[np.concatenate([future_pred, [last_sequence[0, -1, 1], last_sequence[0, -1, 2]]])]], axis=1)

    future_predictions = np.array(future_predictions).reshape(-1, 1)

    # Inverse transform future predictions if data was normalized
    if normalize_data:
        future_predictions = scaler.inverse_transform(np.concatenate([future_predictions, np.zeros((future_predictions.shape[0], 2))], axis=1))[:, 0]

    # Combine past and future predictions
    # complete_estimated_data = np.concatenate([y_train_pred.flatten(), future_predictions.flatten()])

    complete_estimation =  [ *y_train_pred, *future_predictions]
    
    # Error metrics for past predictions
    error_of_past_estimation = np.abs(y_train - y_train_pred.flatten())
    mean_error_past_estimation = np.mean(error_of_past_estimation)
    std_error_past_estimation = np.std(error_of_past_estimation)
    rmse_error_past_estimation = np.sqrt(mean_squared_error(y_train, y_train_pred.flatten()))

    # Error metrics for future predictions, if test sequences are valid
    if X_test.shape[0] > 0:
        future_actual = future_data_values[:, 0]
        error_of_future_estimation = np.abs(future_actual - y_test_pred.flatten())
        mean_error_future_estimation = np.mean(error_of_future_estimation)
        std_error_future_estimation = np.std(error_of_future_estimation)
        rmse_error_future_estimation = np.sqrt(mean_squared_error(future_actual, y_test_pred.flatten()))
    else:
        future_actual = []
        error_of_future_estimation = []
        mean_error_future_estimation = None
        std_error_future_estimation = None
        rmse_error_future_estimation = None
    
    complete_data = data['Error'][:-sequence_length]
    complete_dates = data['Date'][:-sequence_length]
    # print(y_train_pred)
    # print(future_predictions)
    # print(complete_estimation)

    
    
    error_of_complete_estimation = abs(complete_data - complete_estimation)
    mean_error_complete_estimation = np.mean(error_of_complete_estimation)
    std_error_complete_estimation = np.std(error_of_complete_estimation)
    rmse_error_complete_estimation = root_mean_squared_error(complete_data, complete_estimation)
    print(y_test_pred)
    result = {
        link: {
            "Normalized_data": normalize_data,

            "Complete_dates": complete_dates,
            "Complete_data": complete_data,
            "Complete_estimated_data": complete_estimation,

            "Complete_estimation_error": error_of_complete_estimation,
            "Complete_mean_estimation_error": mean_error_complete_estimation,
            "Complete_std_estimation_error": std_error_complete_estimation,
            "Complete_rmse_estimation_error": rmse_error_complete_estimation,

            "Past_dates": past_dates,
            "Last_date": past_dates.iloc[-1].strftime('%Y-%m-%d'),
            "Past_data": data['Error'][:-days_to_remove],
            "Past_estimated_data": y_train_pred.flatten(),

            "Future_dates": future_dates,
            "Future_data": data['Error'][-days_to_remove:],
            "Future_estimated_data": y_test_pred.flatten(),

            "Past_estimation_error": error_of_past_estimation,
            "Past_mean_estimation_error": mean_error_past_estimation,
            "Past_std_estimation_error": std_error_past_estimation,
            "Past_rmse_estimation_error": rmse_error_past_estimation,

            "Future_estimation_error": error_of_future_estimation,
            "Future_mean_estimation_error": mean_error_future_estimation,
            "Future_std_estimation_error": std_error_future_estimation,
            "Future_rmse_estimation_error": rmse_error_future_estimation
        }
    }
    

    if show_figs or save_figures:
        # Plotting

        plt.figure(figsize=(18, 30))


        #Future data plot
        plt.subplot(9, 1, 4)
        plt.plot(result[link]['Future_dates'], result[link]['Future_data'], label='Actual Data', color='royalblue',  marker='o')
        plt.plot(result[link]['Future_dates'], result[link]['Future_estimated_data'], label='Estimated Data', color='darkorange', linestyle='--', marker='x')
        plt.title(f'Actual vs Estimated Data for the future {days_to_remove} Days')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.grid()

        #Future error estimation plot
        plt.subplot(9, 1, 5)
        plt.plot(result[link]['Future_dates'], result[link]['Future_estimation_error'], label='Error of the Estimation', color='green',  marker='o')
        plt.axhline(result[link]['Future_mean_estimation_error'], linestyle='dashed', label='Average Estimation Error', color='purple')
        plt.axhline(result[link]['Future_rmse_estimation_error'], linestyle='dashed', label='Root Mean Squared Error', color='deeppink')
        plt.title(f'Prediction Error of Actual vs Estimated Data for the future {days_to_remove} Days     Avg: {result[link]['Future_mean_estimation_error']:.5f} RMSE:{result[link]['Future_rmse_estimation_error']:.5f}')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.grid()

        # Future upper and lower bounds
        plt.subplot(9, 1, 6)
        plt.plot(result[link]['Future_dates'], result[link]['Future_data']+result[link]['Future_mean_estimation_error'], label='Upper Data Bound', color='teal',  marker='o')
        plt.plot(result[link]['Future_dates'], result[link]['Future_data']-result[link]['Future_mean_estimation_error'], label='Lower Data Bound', color='orchid',  marker='o')
        plt.plot(result[link]['Future_dates'], result[link]['Future_estimated_data'], label='Estimated Data', color='chocolate', linestyle='--',  marker='x')
        plt.title(f'Bounds of Actual vs Estimated Data for the future {days_to_remove} Days   std: {result[link]['Future_std_estimation_error']}')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.grid()


        #Past data plot
        plt.subplot(9, 1, 7)
        plt.plot(result[link]['Past_dates'], result[link]['Past_data'], label='Actual Data', color='royalblue')
        plt.plot(result[link]['Past_dates'], result[link]['Past_estimated_data'], label='Estimated Data', color='darkorange', linestyle='--')
        plt.title(f'Actual vs Estimated Data for the past {N-days_to_remove} Days')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.grid()

        #Past error estimation plot
        plt.subplot(9, 1, 8)
        plt.plot(result[link]['Past_dates'], result[link]['Past_estimation_error'], label='Error of the Estimation', color='green')
        plt.axhline(result[link]['Past_mean_estimation_error'], linestyle='dashed', label='Average Estimation Error', color='purple')
        plt.axhline(result[link]['Past_rmse_estimation_error'], linestyle='dashed', label='Root Mean Squared Error', color='deeppink')
        plt.title(f'Prediction Error of Actual vs Estimated Data for the past {N-days_to_remove} Days     Avg: {result[link]['Past_mean_estimation_error']:.5f}  RMSE:{result[link]['Past_rmse_estimation_error']:.5f}')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.grid()

        # Past upper and lower bounds
        plt.subplot(9, 1, 9)
        plt.plot(result[link]['Past_dates'], result[link]['Past_data']+result[link]['Past_mean_estimation_error'], label='Upper Data Bound', color='teal')
        plt.plot(result[link]['Past_dates'], result[link]['Past_data']-result[link]['Past_mean_estimation_error'], label='Lower Data Bound', color='orchid')
        plt.plot(result[link]['Past_dates'], result[link]['Past_estimated_data'], label='Estimated Data', color='chocolate', linestyle='--')
        plt.title(f'Bounds of Actual vs Estimated Data for the past {N-days_to_remove} Days   std: {result[link]['Past_std_estimation_error']}')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.grid()


        plt.tight_layout()
        if save_figures:
            plt.savefig(save_path)
            plt.close()
        
        if show_figs: 
            plt.show()

    return result


In [ ]:
file_path = 'up_to_15-07-2024/brisbane_0-14.xlsx'  # Update this path to the correct location of your Excel file
df = pd.read_excel(file_path)
lstm_estimator_past(df, 10, 10, '0-14', normalize_data=True, epochs=10, save_figures=False, save_path=None, show_figs=True)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional, LSTM, Dense
from tensorflow.keras.optimizers import Adam

def bilstm_estimator_past(data, N_remove, N_estimate, link, normalize_data=True, epochs=50, save_figures=False, save_path=None, show_figs=False):
    print(f'Estimating for link {link}')

    # Extract the error values and new features
    dates = data['Date']
    # Feature engineering: Extract day of the week and month from the 'Date' column
    data['DayOfWeek'] = data['Date'].dt.dayofweek  # Monday=0, Sunday=6
    data['Month'] = data['Date'].dt.month  # Extracting month
    
    # Prepare the data for model input (including new features)
    feature_columns = ['Error', 'DayOfWeek', 'Month']
    data_features = data[feature_columns]

    # Normalize data if requested (using RobustScaler for outlier resistance)
    if normalize_data:
        scaler = MinMaxScaler(feature_range=(0, 1))
        data_values = scaler.fit_transform(data_features)
    else:
        data_values = data_features.values

    # Number of samples
    N = len(data_values)
    
    # Split the data
    if N_remove == 0:
        past_data_values = data_values
        past_dates = dates
        last_date = past_dates.iloc[-1]
        future_dates = pd.date_range(start=last_date, periods=N_estimate + 1, freq='D')[1:]
    else: 
        past_data_values = data_values[:-N_remove] # All the error values except for the last N_remove days
        future_data_values = data_values[-N_remove:] # The last N_remove error values
        past_dates = dates[:-N_remove] # All the error values except for the last N_remove days
        future_dates = pd.date_range(start=dates.iloc[-N_remove], periods=N_estimate + 1, freq='D')[1:]
    
    # Create sequences for LSTM (including all features)
    sequence_length = 60
    def create_sequences(data, sequence_length):
        sequences = []
        labels = []
        for i in range(sequence_length, len(data)):
            sequences.append(data[i-sequence_length:i])
            labels.append(data[i, 0])  # Predicting the 'Error' value (first column)
        return np.array(sequences), np.array(labels)
    
    # Number of past days to use to predict the next
    X, y = create_sequences(past_data_values, sequence_length)

    # Split into training and testing data
    split = int(0.8 * len(X))  # 80% train, 20% test
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    # Reshape for LSTM (samples, time steps, features)
    X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], X_train.shape[2]))
    X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], X_test.shape[2]))

    # Define the BiLSTM model
    model = Sequential()
    model.add(Bidirectional(LSTM(150, return_sequences=True), input_shape=(X_train.shape[1], X_train.shape[2])))
    model.add(Bidirectional(LSTM(150)))
    model.add(Dense(1))

    # Compile the model
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')

    # Train the model
    history = model.fit(X_train, y_train, epochs=epochs, batch_size=32, validation_data=(X_test, y_test), verbose=1)

    # Predict past data
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Inverse transform if data was normalized
    if normalize_data:
        y_train_pred = scaler.inverse_transform(np.concatenate([y_train_pred, X_train[:, -1, 1:]], axis=1))[:, 0]
        y_test_pred = scaler.inverse_transform(np.concatenate([y_test_pred, X_test[:, -1, 1:]], axis=1))[:, 0]
        y_train = scaler.inverse_transform(np.concatenate([y_train.reshape(-1, 1), X_train[:, -1, 1:]], axis=1))[:, 0]
        y_test = scaler.inverse_transform(np.concatenate([y_test.reshape(-1, 1), X_test[:, -1, 1:]], axis=1))[:, 0]
    else:
        y_train_pred = y_train_pred
        y_test_pred = y_test_pred

    last_sequence = past_data_values[-sequence_length:].reshape((1, sequence_length, past_data_values.shape[1]))
    future_predictions = []

    for _ in range(N_estimate):
        future_pred = model.predict(last_sequence)[0]  # Make prediction

        if future_pred.size == 0:  # Check if prediction is valid
            break  # Avoid continuing if predictions are not generated

        future_predictions.append(future_pred)  # Append prediction to future_predictions

        # Update last_sequence for the next prediction step
        last_sequence = np.append(last_sequence[:, 1:, :], [[np.concatenate([future_pred, [last_sequence[0, -1, 1], last_sequence[0, -1, 2]]])]], axis=1)
        
    # Check if future_predictions contains data
    if len(future_predictions) == 0:
        raise ValueError("No future predictions were generated.")

    future_predictions = np.array(future_predictions).reshape(-1, 1)

    # Inverse transform future predictions if data was normalized
    if normalize_data:
        future_predictions = scaler.inverse_transform(np.concatenate([future_predictions, np.zeros((future_predictions.shape[0], 2))], axis=1))[:, 0]

    # Combine train and test predictions
    complete_dates = pd.concat([pd.Series(dates[:-N_remove]), pd.Series(future_dates)])
    complete_estimated_data = np.concatenate([model.predict(X).flatten(), future_predictions.flatten()])

    # Error metrics
    error_of_train_estimation = np.abs(y_train - y_train_pred)
    mean_error_train_estimation = np.mean(error_of_train_estimation)
    std_error_train_estimation = np.std(error_of_train_estimation)
    rmse_error_train_estimation = np.sqrt(mean_squared_error(y_train, y_train_pred))

    error_of_test_estimation = np.abs(y_test - y_test_pred)
    mean_error_test_estimation = np.mean(error_of_test_estimation)
    std_error_test_estimation = np.std(error_of_test_estimation)
    rmse_error_test_estimation = np.sqrt(mean_squared_error(y_test, y_test_pred))

    result = {
        link: {
            "Normalized_data": normalize_data,

            "Complete_dates": complete_dates,
            "Complete_estimated_data": complete_estimated_data.tolist(),
                       
            "Past_dates": dates,
            "Last_date": dates.iloc[-1],
            "Past_data": data['Error'].values.tolist(),

            "Future_dates": future_dates,
            "Future_estimated_data": future_predictions.tolist(),

            "Train_dates": dates[:split + sequence_length],
            "Last_train_date": dates[split + sequence_length - 1],
            "Train_data": y_train.tolist(),
            "Train_estimated_data": y_train_pred.tolist(),

            "Train_estimation_error": error_of_train_estimation.tolist(),
            "Train_mean_estimation_error": mean_error_train_estimation,
            "Train_std_estimation_error": std_error_train_estimation,
            "Train_rmse_estimation_error": rmse_error_train_estimation,

            "Test_dates": dates[split + sequence_length:],
            "Test_data": y_test.tolist(),
            "Test_estimated_data": y_test_pred.tolist(),

            "Test_estimation_error": error_of_test_estimation.tolist(),
            "Test_mean_estimation_error": mean_error_test_estimation,
            "Test_std_estimation_error": std_error_test_estimation,
            "Test_rmse_estimation_error": rmse_error_test_estimation,
        }
    }

    # Plotting
    if show_figs or save_figures:
        plt.figure(figsize=(18, 30))

        # Train data plot
        plt.subplot(4, 1, 1)
        plt.plot(result[link]['Train_dates'][:len(result[link]['Train_data'])], result[link]['Train_data'], label='Actual Data', color='royalblue')
        plt.plot(result[link]['Train_dates'][:len(result[link]['Train_estimated_data'])], result[link]['Train_estimated_data'], label='Estimated Data', color='darkorange', linestyle='--')
        plt.title(f'Actual vs Estimated Data (train)')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.grid()

        # Test data plot
        plt.subplot(4, 1, 2)
        plt.plot(result[link]['Test_dates'][:len(result[link]['Test_data'])], result[link]['Test_data'], label='Actual Data', color='royalblue')
        plt.plot(result[link]['Test_dates'][:len(result[link]['Test_estimated_data'])], result[link]['Test_estimated_data'], label='Estimated Data', color='darkorange', linestyle='--')
        plt.title(f'Actual vs Estimated Data (test)  RMSE: {result[link]["Test_rmse_estimation_error"]}')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.grid()

        # Upper and lower bounds for test data
        plt.subplot(4, 1, 3)
        plt.plot(result[link]['Test_dates'][:len(result[link]['Test_data'])], np.array(result[link]['Test_data']) + result[link]['Test_mean_estimation_error'], label='Upper Data Bound', color='teal')
        plt.plot(result[link]['Test_dates'][:len(result[link]['Test_data'])], np.array(result[link]['Test_data']) - result[link]['Test_mean_estimation_error'], label='Lower Data Bound', color='orchid')
        plt.plot(result[link]['Test_dates'][:len(result[link]['Test_estimated_data'])], result[link]['Test_estimated_data'], label='Estimated Data', color='chocolate', linestyle='--')
        plt.title(f'Bounds of Actual vs Estimated Data (test) std: {result[link]["Test_std_estimation_error"]}')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.grid()

        # Future data plot
        # plt.subplot(4, 1, 4)
        # plt.plot(result[link]['Future_dates'][:len(result[link]['Future_estimated_data'])], result[link]['Future_estimated_data'], label='Predicted Data', color='darkorange', linestyle='--')
        # plt.title(f'Future Prediction')
        # plt.xlabel('Date')
        # plt.ylabel('Error')
        # plt.legend()
        # plt.grid()

        plt.tight_layout()
        if save_figures:
            plt.savefig(save_path)

        if show_figs:
            plt.show()

        plt.close()

    return result
